In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from pathlib import Path
import matplotlib.pyplot as plt
import json

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"
FIG_DIR  = BASE_DIR / "results" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

master = pd.read_csv(PROC_DIR / "training_table.csv")

all_cols    = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_')
               or c.startswith('Nuclei_')
               or c.startswith('Cytoplasm_')]
ic50_cols   = [c for c in all_cols
               if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

drug_names  = master['drug_name'].values
drug_index  = {name: i for i, name in enumerate(drug_names)}

# Train/val/test split — same seed as notebook 04
np.random.seed(42)
shuffled_idx = np.random.permutation(len(drug_names))
n_test  = int(len(drug_names) * 0.15)
n_val   = int(len(drug_names) * 0.15)
n_train = len(drug_names) - n_test - n_val

train_drugs = set(drug_names[shuffled_idx[:n_train]])
val_drugs   = set(drug_names[shuffled_idx[n_train:n_train+n_val]])
test_drugs  = set(drug_names[shuffled_idx[n_train+n_val:]])

ic50_long = master[['drug_name'] + ic50_cols].melt(
    id_vars='drug_name', var_name='cell_line', value_name='ln_ic50'
).dropna(subset=['ln_ic50']).reset_index(drop=True)

train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
val_df   = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
test_df  = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

# Normalize morphology on train only
morgan_lookup = master.set_index('drug_name')[morgan_cols].values
morpho_lookup = master.set_index('drug_name')[morpho_cols].values
train_idx     = [drug_index[d] for d in train_drugs]

morpho_scaler  = StandardScaler()
morpho_scaled  = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
morgan_scaled  = morgan_lookup.astype(np.float32)

print(f"Train: {len(train_drugs)} drugs | {len(train_df):,} pairs")
print(f"Val:   {len(val_drugs)} drugs | {len(val_df):,} pairs")
print(f"Test:  {len(test_drugs)} drugs | {len(test_df):,} pairs")
print(f"\nMorgan dim:    {morgan_scaled.shape[1]}")
print(f"Morphology dim:{morpho_scaled.shape[1]}")

Train: 123 drugs | 103,319 pairs
Val:   26 drugs | 22,677 pairs
Test:  26 drugs | 23,683 pairs

Morgan dim:    2048
Morphology dim:3178


In [2]:
# cross-attention fuison architecture

class CrossAttentionFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178, 
                 embed_dim=256, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        
        # Project each modality into shared embedding space
        self.morgan_proj = nn.Sequential(
            nn.Linear(morgan_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.morpho_proj = nn.Sequential(
            nn.Linear(morpho_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Cross-attention: morphology attends to structure
        # Query = morphology, Key/Value = structure
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(embed_dim, num_heads, 
                                  dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(embed_dim) for _ in range(num_layers)
        ])
        
        # MLP readout
        self.readout = nn.Sequential(
            nn.Linear(embed_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, morgan, morpho):
        # Project to embedding space
        # Shape: (batch, embed_dim)
        z_morgan = self.morgan_proj(morgan)
        z_morpho = self.morpho_proj(morpho)
        
        # Reshape for attention: (batch, seq_len=1, embed_dim)
        q = z_morpho.unsqueeze(1)
        k = z_morgan.unsqueeze(1)
        v = z_morgan.unsqueeze(1)
        
        # Cross-attention layers with residual
        attn_out = q
        for attn_layer, norm in zip(self.cross_attn_layers, self.layer_norms):
            attended, _ = attn_layer(attn_out, k, v)
            attn_out = norm(attn_out + attended)
        
        # Squeeze back: (batch, embed_dim)
        attn_out = attn_out.squeeze(1)
        
        # Concatenate attended morphology with original structure embedding
        fused = torch.cat([attn_out, z_morgan], dim=1)
        
        return self.readout(fused).squeeze(-1)


# Parameter count
model_attn = CrossAttentionFusion()
n_params = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)
print(f"CrossAttentionFusion architecture:")
print(f"  Morgan projection:  {2048} → 256")
print(f"  Morpho projection:  {3178} → 256")
print(f"  Cross-attention:    {4} heads × {2} layers")
print(f"  Readout MLP:        512 → 256 → 64 → 1")
print(f"  Total parameters:   {n_params:,}")

CrossAttentionFusion architecture:
  Morgan projection:  2048 → 256
  Morpho projection:  3178 → 256
  Cross-attention:    4 heads × 2 layers
  Readout MLP:        512 → 256 → 64 → 1
  Total parameters:   2,015,233


In [5]:
# dataset and training loop

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_scaled, morpho_scaled):
        self.drug_idx = torch.tensor(
            [drug_index[d] for d in df['drug_name']], dtype=torch.long
        )
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan  = torch.tensor(morgan_scaled, dtype=torch.float32)
        self.morpho  = torch.tensor(morpho_scaled, dtype=torch.float32)

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]


train_ds = DrugResponseDataset(train_df, drug_index, morgan_scaled, morpho_scaled)
val_ds   = DrugResponseDataset(val_df,   drug_index, morgan_scaled, morpho_scaled)
test_ds  = DrugResponseDataset(test_df,  drug_index, morgan_scaled, morpho_scaled)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=512, shuffle=False, num_workers=0)


def train_cross_attention(model, train_loader, val_loader,
                          epochs=150, lr=1e-3, patience=20, weight_decay=1e-4):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )
    criterion = nn.MSELoss()

    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0
    history          = {'train': [], 'val': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            out  = model(morgan_b, morpho_b)
            loss = criterion(out, target_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(target_b)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                out       = model(morgan_b, morpho_b)
                val_loss += criterion(out, target_b).item() * len(target_b)
        val_loss /= len(val_loader.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        if (epoch+1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d} | train: {train_loss:.4f} | val: {val_loss:.4f}")

    model.load_state_dict(best_state)
    return model, history


def evaluate(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for morgan_b, morpho_b, target_b in loader:
            preds.extend(model(morgan_b, morpho_b).numpy())
            targets.extend(target_b.numpy())
    preds   = np.array(preds)
    targets = np.array(targets)
    r, _    = pearsonr(preds, targets)
    rmse    = np.sqrt(((preds - targets)**2).mean())
    return r, rmse, preds, targets

print("Dataset and training loop ready.")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")

Dataset and training loop ready.
Train batches: 202
Val batches:   45


In [8]:
# redesigned cross-attention with tokenization

class CrossAttentionFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178,
                 token_dim=64, num_tokens=16, num_heads=4, 
                 num_layers=2, dropout=0.3):
        super().__init__()
        self.num_tokens = num_tokens
        self.token_dim  = token_dim

        # Split Morgan (2048) into 16 tokens of 128 → project to token_dim
        self.morgan_token_proj = nn.Linear(morgan_dim // num_tokens, token_dim)

        # Split Morphology into 16 tokens → project to token_dim
        # Pad morpho to be divisible by num_tokens
        self.morpho_pad = (num_tokens - (morpho_dim % num_tokens)) % num_tokens
        morpho_padded   = morpho_dim + self.morpho_pad
        self.morpho_token_proj = nn.Linear(morpho_padded // num_tokens, token_dim)

        # Cross-attention layers: morphology tokens attend to structure tokens
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(token_dim, num_heads,
                                  dropout=dropout, batch_first=True)
            for _ in range(num_layers)
        ])
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(token_dim) for _ in range(num_layers)
        ])

        # Readout: pool attended tokens → MLP
        self.readout = nn.Sequential(
            nn.Linear(token_dim * num_tokens * 2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

        self.morgan_dim = morgan_dim
        self.morpho_dim = morpho_dim

    def tokenize(self, x, proj, n_tokens, pad=0):
        if pad > 0:
            x = F.pad(x, (0, pad))
        batch = x.shape[0]
        # (batch, n_tokens, token_size)
        x = x.view(batch, n_tokens, -1)
        return proj(x)  # (batch, n_tokens, token_dim)

    def forward(self, morgan, morpho):
        # Tokenize
        z_morgan = self.tokenize(morgan, self.morgan_token_proj,
                                 self.num_tokens)
        z_morpho = self.tokenize(morpho, self.morpho_token_proj,
                                 self.num_tokens, pad=self.morpho_pad)

        # Cross-attention: morphology queries, structure keys/values
        q = z_morpho
        k = z_morgan
        v = z_morgan

        attn_out = q
        for attn_layer, norm in zip(self.cross_attn_layers, self.layer_norms):
            attended, _ = attn_layer(attn_out, k, v)
            attn_out    = norm(attn_out + attended)

        # Flatten tokens from both streams
        fused = torch.cat([
            attn_out.reshape(attn_out.shape[0], -1),
            z_morgan.reshape(z_morgan.shape[0], -1)
        ], dim=1)

        return self.readout(fused).squeeze(-1)


torch.manual_seed(42)
model_attn = CrossAttentionFusion()
n_params = sum(p.numel() for p in model_attn.parameters() if p.requires_grad)
print(f"Redesigned CrossAttentionFusion:")
print(f"  Morgan tokens:    16 × 128-dim → 16 × 64-dim")
print(f"  Morpho tokens:    16 × ~200-dim → 16 × 64-dim")
print(f"  Cross-attention:  4 heads × 2 layers")
print(f"  Fused dim:        16×64 + 16×64 = 2048 → 256 → 64 → 1")
print(f"  Total parameters: {n_params:,}")

# Quick forward pass sanity check
dummy_morgan = torch.randn(4, 2048)
dummy_morpho = torch.randn(4, 3178)
out = model_attn(dummy_morgan, dummy_morpho)
print(f"\nSanity check output shape: {out.shape}")
print("Architecture valid.")

Redesigned CrossAttentionFusion:
  Morgan tokens:    16 × 128-dim → 16 × 64-dim
  Morpho tokens:    16 × ~200-dim → 16 × 64-dim
  Cross-attention:  4 heads × 2 layers
  Fused dim:        16×64 + 16×64 = 2048 → 256 → 64 → 1
  Total parameters: 596,289

Sanity check output shape: torch.Size([4])
Architecture valid.


In [9]:
# traning

print("="*50)
print("Cross-Attention Fusion Model (Tokenized)")
print("="*50)

torch.manual_seed(42)
model_attn = CrossAttentionFusion()

print("\nTraining...")
model_attn, hist_attn = train_cross_attention(
    model_attn, train_loader, val_loader,
    epochs=150, lr=1e-3, patience=20, weight_decay=1e-4
)

r_attn, rmse_attn, preds_attn, targets_attn = evaluate(model_attn, test_loader)

print(f"\n{'='*50}")
print(f"RESULTS SUMMARY")
print(f"{'='*50}")
print(f"  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652")
print(f"  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883")
print(f"  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542")
print(f"  Cross-Attention | Pearson R: {r_attn:.4f} | RMSE: {rmse_attn:.4f}")

Cross-Attention Fusion Model (Tokenized)

Training...
  Epoch  10 | train: 2.3972 | val: 3.3990
  Epoch  20 | train: 2.3592 | val: 3.4674
  Early stopping at epoch 24

RESULTS SUMMARY
  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652
  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883
  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542
  Cross-Attention | Pearson R: 0.5972 | RMSE: 2.0656


In [10]:
# train with warmup scheduler

def train_with_warmup(model, train_loader, val_loader,
                      epochs=150, lr_max=1e-3, warmup_epochs=10,
                      patience=25, weight_decay=1e-4):
    optimizer = torch.optim.Adam(
        model.parameters(), lr=1e-4, weight_decay=weight_decay
    )
    criterion = nn.MSELoss()

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return max(0.1, 0.5 ** ((epoch - warmup_epochs) // 20))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    best_val_loss    = float('inf')
    best_state       = None
    patience_counter = 0
    history          = {'train': [], 'val': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            out  = model(morgan_b, morpho_b)
            loss = criterion(out, target_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * len(target_b)
        train_loss /= len(train_loader.dataset)
        scheduler.step()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                out       = model(morgan_b, morpho_b)
                val_loss += criterion(out, target_b).item() * len(target_b)
        val_loss /= len(val_loader.dataset)

        history['train'].append(train_loss)
        history['val'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_state       = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

        if (epoch+1) % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1:3d} | train: {train_loss:.4f} | val: {val_loss:.4f} | lr: {current_lr:.6f}")

    model.load_state_dict(best_state)
    return model, history

torch.manual_seed(42)
model_attn_v2 = CrossAttentionFusion()

print("Training with warmup scheduler...")
model_attn_v2, hist_attn_v2 = train_with_warmup(
    model_attn_v2, train_loader, val_loader,
    epochs=150, lr_max=1e-3, warmup_epochs=10, patience=25
)

r_v2, rmse_v2, _, _ = evaluate(model_attn_v2, test_loader)

print(f"\n{'='*50}")
print(f"RESULTS SUMMARY")
print(f"{'='*50}")
print(f"  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652")
print(f"  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883")
print(f"  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542")
print(f"  Cross-Attn v1   | Pearson R: 0.5972 | RMSE: 2.0656")
print(f"  Cross-Attn v2   | Pearson R: {r_v2:.4f} | RMSE: {rmse_v2:.4f}")

Training with warmup scheduler...
  Epoch  10 | train: 2.6372 | val: 3.9823 | lr: 0.000100
  Epoch  20 | train: 2.4509 | val: 3.5172 | lr: 0.000100
  Epoch  30 | train: 2.4249 | val: 3.3424 | lr: 0.000050
  Epoch  40 | train: 2.4027 | val: 3.3994 | lr: 0.000050
  Epoch  50 | train: 2.3985 | val: 3.3904 | lr: 0.000025
  Early stopping at epoch 55

RESULTS SUMMARY
  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652
  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883
  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542
  Cross-Attn v1   | Pearson R: 0.5972 | RMSE: 2.0656
  Cross-Attn v2   | Pearson R: 0.5887 | RMSE: 2.0938


In [12]:
# gated fusion model

class GatedFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178, 
                 embed_dim=256, dropout=0.3):
        super().__init__()

        # Project each modality to shared space
        self.morgan_proj = nn.Sequential(
            nn.Linear(morgan_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.morpho_proj = nn.Sequential(
            nn.Linear(morpho_dim, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Gating: learn how much to trust each modality
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, 2),
            nn.Softmax(dim=1)
        )

        # Readout
        self.readout = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, morgan, morpho):
        z_morgan = self.morgan_proj(morgan)
        z_morpho = self.morpho_proj(morpho)

        # Compute gates — how much weight to give each modality
        combined = torch.cat([z_morgan, z_morpho], dim=1)
        gates    = self.gate(combined)  # (batch, 2)

        # Weighted combination
        fused = gates[:, 0:1] * z_morgan + gates[:, 1:2] * z_morpho

        return self.readout(fused).squeeze(-1)


torch.manual_seed(42)
model_gated = GatedFusion()
n_params = sum(p.numel() for p in model_gated.parameters() if p.requires_grad)

# Sanity check
out = model_gated(torch.randn(4, 2048), torch.randn(4, 3178))
print(f"GatedFusion:")
print(f"  Morgan proj:  2048 → 256")
print(f"  Morpho proj:  3178 → 256")
print(f"  Gate:         512 → 256 → 2 (softmax weights)")
print(f"  Readout:      256 → 128 → 64 → 1")
print(f"  Parameters:   {n_params:,}")
print(f"  Output shape: {out.shape}")

GatedFusion:
  Morgan proj:  2048 → 256
  Morpho proj:  3178 → 256
  Gate:         512 → 256 → 2 (softmax weights)
  Readout:      256 → 128 → 64 → 1
  Parameters:   1,512,835
  Output shape: torch.Size([4])


In [13]:
# training gated fusion model

print("="*50)
print("Gated Fusion Model")
print("="*50)

torch.manual_seed(42)
model_gated = GatedFusion()

print("\nTraining...")
model_gated, hist_gated = train_cross_attention(
    model_gated, train_loader, val_loader,
    epochs=150, lr=1e-3, patience=20, weight_decay=1e-4
)

r_gated, rmse_gated, _, _ = evaluate(model_gated, test_loader)

print(f"\n{'='*50}")
print(f"FULL RESULTS SUMMARY")
print(f"{'='*50}")
print(f"  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652")
print(f"  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883")
print(f"  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542")
print(f"  Cross-Attention | Pearson R: 0.5972 | RMSE: 2.0656")
print(f"  Gated Fusion    | Pearson R: {r_gated:.4f} | RMSE: {rmse_gated:.4f}")

Gated Fusion Model

Training...
  Epoch  10 | train: 2.4114 | val: 3.9461
  Epoch  20 | train: 2.3837 | val: 4.0966
  Early stopping at epoch 22

FULL RESULTS SUMMARY
  Structure-only  | Pearson R: 0.3333 | RMSE: 2.4652
  Morphology-only | Pearson R: 0.6002 | RMSE: 2.0883
  Concatenation   | Pearson R: 0.6085 | RMSE: 2.0542
  Cross-Attention | Pearson R: 0.5972 | RMSE: 2.0656
  Gated Fusion    | Pearson R: 0.5848 | RMSE: 2.0561


In [17]:
# save results & models

import json

final_results = {
    'structure_only':  {'pearson_r': 0.3333, 'rmse': 2.4652},
    'morphology_only': {'pearson_r': 0.6002, 'rmse': 2.0883},
    'concatenation':   {'pearson_r': 0.6085, 'rmse': 2.0542},
    'cross_attention': {'pearson_r': 0.5972, 'rmse': 2.0656},
    'gated_fusion':    {'pearson_r': 0.5848, 'rmse': 2.0561}
}

(BASE_DIR / "results").mkdir(parents=True, exist_ok=True)

with open(BASE_DIR / "results" / "all_results.json", 'w') as f:
    json.dump(final_results, f, indent=2)

torch.save(model_gated.state_dict(), BASE_DIR / "results" / "model_gated.pt")
torch.save(model_attn.state_dict(),  BASE_DIR / "results" / "model_attn.pt")

print("Saved: all_results.json")
print("Saved: model_gated.pt, model_attn.pt")
print("\nKey finding: morphology +80% over structure.")
print("Fusion methods limited by dataset scale.")

Saved: all_results.json
Saved: model_gated.pt, model_attn.pt

Key finding: morphology +80% over structure.
Fusion methods limited by dataset scale.


In [1]:
# Multi-seed evaluation for cross-attention and gated fusion models
# Mirrors the multi-seed protocol already used for structure-only, morphology-only,
# and concatenation, using identical data splits and seeds for a fair comparison.

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from pathlib import Path
import json

BASE_DIR = Path("..")
PROC_DIR = BASE_DIR / "data" / "processed"

master = pd.read_csv(PROC_DIR / "training_table.csv")
all_cols = master.columns.tolist()
morgan_cols = [c for c in all_cols if c.startswith('morgan_')]
morpho_cols = [c for c in all_cols if c.startswith('Cells_') or c.startswith('Nuclei_') or c.startswith('Cytoplasm_')]
ic50_cols = [c for c in all_cols if c not in morgan_cols + morpho_cols + ['drug_name', 'Metadata_JCP2022']]

drug_names = master['drug_name'].values
drug_index = {name: i for i, name in enumerate(drug_names)}
morgan_lookup = master.set_index('drug_name')[morgan_cols].values.astype(np.float32)
morpho_lookup = master.set_index('drug_name')[morpho_cols].values

ic50_long = master[['drug_name'] + ic50_cols].melt(
    id_vars='drug_name', var_name='cell_line', value_name='ln_ic50'
).dropna(subset=['ln_ic50']).reset_index(drop=True)

class DrugResponseDataset(Dataset):
    def __init__(self, df, drug_index, morgan_scaled, morpho_scaled):
        self.drug_idx = torch.tensor([drug_index[d] for d in df['drug_name']], dtype=torch.long)
        self.targets = torch.tensor(df['ln_ic50'].values, dtype=torch.float32)
        self.morgan = torch.tensor(morgan_scaled, dtype=torch.float32)
        self.morpho = torch.tensor(morpho_scaled, dtype=torch.float32)
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, i):
        idx = self.drug_idx[i]
        return self.morgan[idx], self.morpho[idx], self.targets[i]

class CrossAttentionFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178, token_dim=64, num_tokens=16, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        self.num_tokens = num_tokens
        self.morgan_token_proj = nn.Linear(morgan_dim // num_tokens, token_dim)
        self.morpho_pad = (num_tokens - (morpho_dim % num_tokens)) % num_tokens
        morpho_padded = morpho_dim + self.morpho_pad
        self.morpho_token_proj = nn.Linear(morpho_padded // num_tokens, token_dim)
        self.cross_attn_layers = nn.ModuleList([nn.MultiheadAttention(token_dim, num_heads, dropout=dropout, batch_first=True) for _ in range(num_layers)])
        self.layer_norms = nn.ModuleList([nn.LayerNorm(token_dim) for _ in range(num_layers)])
        self.readout = nn.Sequential(
            nn.Linear(token_dim * num_tokens * 2, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    def tokenize(self, x, proj, n_tokens, pad=0):
        if pad > 0:
            x = F.pad(x, (0, pad))
        batch = x.shape[0]
        x = x.view(batch, n_tokens, -1)
        return proj(x)
    def forward(self, morgan, morpho):
        z_morgan = self.tokenize(morgan, self.morgan_token_proj, self.num_tokens)
        z_morpho = self.tokenize(morpho, self.morpho_token_proj, self.num_tokens, pad=self.morpho_pad)
        q, k, v = z_morpho, z_morgan, z_morgan
        attn_out = q
        for attn_layer, norm in zip(self.cross_attn_layers, self.layer_norms):
            attended, _ = attn_layer(attn_out, k, v)
            attn_out = norm(attn_out + attended)
        fused = torch.cat([attn_out.reshape(attn_out.shape[0], -1), z_morgan.reshape(z_morgan.shape[0], -1)], dim=1)
        return self.readout(fused).squeeze(-1)

class GatedFusion(nn.Module):
    def __init__(self, morgan_dim=2048, morpho_dim=3178, embed_dim=256, dropout=0.3):
        super().__init__()
        self.morgan_proj = nn.Sequential(nn.Linear(morgan_dim, embed_dim), nn.LayerNorm(embed_dim), nn.ReLU(), nn.Dropout(dropout))
        self.morpho_proj = nn.Sequential(nn.Linear(morpho_dim, embed_dim), nn.LayerNorm(embed_dim), nn.ReLU(), nn.Dropout(dropout))
        self.gate = nn.Sequential(nn.Linear(embed_dim*2, embed_dim), nn.ReLU(), nn.Linear(embed_dim, 2), nn.Softmax(dim=1))
        self.readout = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    def forward(self, morgan, morpho):
        z_morgan = self.morgan_proj(morgan)
        z_morpho = self.morpho_proj(morpho)
        gates = self.gate(torch.cat([z_morgan, z_morpho], dim=1))
        fused = gates[:, 0:1] * z_morgan + gates[:, 1:2] * z_morpho
        return self.readout(fused).squeeze(-1)

def train_fusion_model(model, train_loader, val_loader, epochs=150, lr=1e-3, patience=20, weight_decay=1e-4, clip=False):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    criterion = nn.MSELoss()
    best_val, best_state, patience_counter = float('inf'), None, 0
    for epoch in range(epochs):
        model.train()
        for morgan_b, morpho_b, target_b in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(morgan_b, morpho_b), target_b)
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for morgan_b, morpho_b, target_b in val_loader:
                val_loss += criterion(model(morgan_b, morpho_b), target_b).item() * len(target_b)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break
    model.load_state_dict(best_state)
    return model

def evaluate_fusion(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for morgan_b, morpho_b, target_b in loader:
            preds.extend(model(morgan_b, morpho_b).numpy())
            targets.extend(target_b.numpy())
    preds, targets = np.array(preds), np.array(targets)
    r, _ = pearsonr(preds, targets)
    rmse = np.sqrt(((preds - targets)**2).mean())
    return r, rmse, preds, targets

SEEDS = [42, 0, 1, 7, 21]
fusion_results = {'cross_attention': {'r': [], 'rmse': [], 'preds': {}}, 'gated_fusion': {'r': [], 'rmse': [], 'preds': {}}}

for seed in SEEDS:
    print(f"\n{'='*50}\nSeed {seed}\n{'='*50}")
    np.random.seed(seed)
    torch.manual_seed(seed)

    shuffled_idx = np.random.permutation(len(drug_names))
    n_test = int(len(drug_names) * 0.15)
    n_val = int(len(drug_names) * 0.15)
    n_train = len(drug_names) - n_test - n_val
    train_drugs = set(drug_names[shuffled_idx[:n_train]])
    val_drugs = set(drug_names[shuffled_idx[n_train:n_train+n_val]])
    test_drugs = set(drug_names[shuffled_idx[n_train+n_val:]])

    train_df = ic50_long[ic50_long['drug_name'].isin(train_drugs)].reset_index(drop=True)
    val_df = ic50_long[ic50_long['drug_name'].isin(val_drugs)].reset_index(drop=True)
    test_df = ic50_long[ic50_long['drug_name'].isin(test_drugs)].reset_index(drop=True)

    train_idx = [drug_index[d] for d in train_drugs]
    morpho_scaler = StandardScaler()
    morpho_scaled = morpho_scaler.fit(morpho_lookup[train_idx]).transform(morpho_lookup)
    morgan_scaled = morgan_lookup

    train_ds = DrugResponseDataset(train_df, drug_index, morgan_scaled, morpho_scaled)
    val_ds = DrugResponseDataset(val_df, drug_index, morgan_scaled, morpho_scaled)
    test_ds = DrugResponseDataset(test_df, drug_index, morgan_scaled, morpho_scaled)

    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

    model_attn = CrossAttentionFusion()
    model_attn = train_fusion_model(model_attn, train_loader, val_loader, clip=True)
    r_attn, rmse_attn, preds_attn, targets_attn = evaluate_fusion(model_attn, test_loader)
    fusion_results['cross_attention']['r'].append(float(r_attn))
    fusion_results['cross_attention']['rmse'].append(float(rmse_attn))
    fusion_results['cross_attention']['preds'][str(seed)] = {'preds': preds_attn.tolist(), 'targets': targets_attn.tolist()}
    print(f"Cross-Attention | R: {r_attn:.4f} | RMSE: {rmse_attn:.4f}")

    model_gated = GatedFusion()
    model_gated = train_fusion_model(model_gated, train_loader, val_loader, clip=False)
    r_gated, rmse_gated, preds_gated, targets_gated = evaluate_fusion(model_gated, test_loader)
    fusion_results['gated_fusion']['r'].append(float(r_gated))
    fusion_results['gated_fusion']['rmse'].append(float(rmse_gated))
    fusion_results['gated_fusion']['preds'][str(seed)] = {'preds': preds_gated.tolist(), 'targets': targets_gated.tolist()}
    print(f"Gated Fusion    | R: {r_gated:.4f} | RMSE: {rmse_gated:.4f}")

print("\n" + "="*50)
print("MULTI-SEED FUSION RESULTS SUMMARY")
print("="*50)
for name in fusion_results:
    r_mean = np.mean(fusion_results[name]['r'])
    r_std = np.std(fusion_results[name]['r'])
    rmse_mean = np.mean(fusion_results[name]['rmse'])
    rmse_std = np.std(fusion_results[name]['rmse'])
    print(f"{name:20s} | R: {r_mean:.4f} +/- {r_std:.4f} | RMSE: {rmse_mean:.4f} +/- {rmse_std:.4f}")
    print(f"  per-seed R: {[round(x,4) for x in fusion_results[name]['r']]}")

with open(BASE_DIR / "results" / "fusion_multiseed_results.json", "w") as f:
    json.dump(fusion_results, f, indent=2)
print("\nSaved: fusion_multiseed_results.json")


Seed 42
Cross-Attention | R: 0.5972 | RMSE: 2.0656
Gated Fusion    | R: 0.5838 | RMSE: 2.0786

Seed 0
Cross-Attention | R: 0.6341 | RMSE: 1.7532
Gated Fusion    | R: 0.5991 | RMSE: 1.8258

Seed 1
Cross-Attention | R: 0.4730 | RMSE: 1.9355
Gated Fusion    | R: 0.4841 | RMSE: 1.9322

Seed 7
Cross-Attention | R: 0.5512 | RMSE: 1.7990
Gated Fusion    | R: 0.4571 | RMSE: 1.9123

Seed 21
Cross-Attention | R: 0.5415 | RMSE: 2.4174
Gated Fusion    | R: 0.4372 | RMSE: 2.6476

MULTI-SEED FUSION RESULTS SUMMARY
cross_attention      | R: 0.5594 +/- 0.0545 | RMSE: 1.9941 +/- 0.2382
  per-seed R: [0.5972, 0.6341, 0.473, 0.5512, 0.5415]
gated_fusion         | R: 0.5122 +/- 0.0665 | RMSE: 2.0793 +/- 0.2956
  per-seed R: [0.5838, 0.5991, 0.4841, 0.4571, 0.4372]

Saved: fusion_multiseed_results.json


In [3]:
# Statistical comparison of fusion strategies (5-seed, paired t-tests)
# Loads the new fusion multi-seed results and compares against concatenation,
# using the same paired t-test approach already used for structure/morphology/concat.

import json
import numpy as np
from scipy.stats import ttest_rel

with open("../results/fusion_multiseed_results.json") as f:
    fusion = json.load(f)
with open("../results/final_results.json") as f:
    final = json.load(f)

r_concat = final['mlp_multiseed']['concatenation']['r_per_seed']
r_attn = fusion['cross_attention']['r']
r_gated = fusion['gated_fusion']['r']

t_attn, p_attn = ttest_rel(r_attn, r_concat)
t_gated, p_gated = ttest_rel(r_gated, r_concat)

print(f"Cross-attention vs Concatenation: t={t_attn:.2f}, p={p_attn:.3f}")
print(f"Gated fusion vs Concatenation:    t={t_gated:.2f}, p={p_gated:.3f}")

Cross-attention vs Concatenation: t=0.83, p=0.451
Gated fusion vs Concatenation:    t=-1.16, p=0.309
